# SAM 3D Body on a free Colab T4

Turns a single photo of a person into a **rigged, untextured** 3D human mesh — the Momentum Human Rig (MHR), ~18.4k vertices, 70 body+hand joints — using Meta's [SAM 3D Body](https://github.com/facebookresearch/sam-3d-body) model ([arXiv 2602.15989](https://arxiv.org/abs/2602.15989), [blog](https://ai.meta.com/blog/sam-3d/), checkpoints released 2025-11-19). This notebook adapts the official [`notebook/demo_human.ipynb`](https://github.com/facebookresearch/sam-3d-body/blob/main/notebook/demo_human.ipynb) from that repo for a stock free-tier Colab T4.

**This is NOT SAM 3D Objects.** SAM 3D Objects is the sibling Meta model that reconstructs *textured* 3D objects/scenes and needs ~32 GB of cloud GPU VRAM. SAM 3D Body is a much smaller, promptable human-mesh-recovery model that fits a free 16 GB T4. The mesh this notebook produces has **no texture/color** — it's geometry + skeleton only.

## Expected runtime (first run on a fresh VM)

| Step | Time |
|---|---|
| Install pip deps + build detectron2 from source | ~8-12 min |
| Download checkpoints (~6.9 GB: body model + ViTDet-H detector + MoGe2 FOV estimator) | ~2-5 min |
| Per-photo inference | ~10-30 sec |
| **Total first run** | **~15-20 min**, then seconds per additional photo |

## Hardware

`Runtime > Change runtime type > T4 GPU` (free tier). Colab Pro / A100 / L4 are **not** required. Based on checkpoint sizes (below), peak VRAM for the full pipeline (body model + ViTDet-H detector + MoGe2) is estimated at roughly 8-11 GB against the T4's ~15 GB usable — **this estimate has not been confirmed on a live run**; if you hit an out-of-memory error see the troubleshooting doc.

## Checkpoint access (gated)

The official weights on Hugging Face require Meta's approval (name, date of birth, country, affiliation, job title — auto- or manually-approved). Request access **before** running this notebook if you want the official copy:

- https://huggingface.co/facebook/sam-3d-body-dinov3 (default here — 840M params, best accuracy)
- https://huggingface.co/facebook/sam-3d-body-vith (631M params — smaller/faster fallback, set `MODEL_VARIANT = "vith"` below)

If access is still pending, the weights cell below automatically falls back to a community re-upload of the same checkpoint files that was confirmed **not gated** via the HF API at the time this notebook was written (gating status can change — this is not guaranteed to stay open):

- https://huggingface.co/jetjodh/sam-3d-body-dinov3
- https://huggingface.co/jetjodh/sam-3d-body-vith

## License

Model + code are under Meta's **SAM License** (a custom Meta research license — NOT MIT/Apache). It permits use, reproduction, distribution and derivative works, with no warranty and no military/ITAR use. Full text: https://github.com/facebookresearch/sam-3d-body/blob/main/LICENSE

## What you get out

Per detected person: a `.obj` mesh (guaranteed), a best-effort `.glb`, and a `_pose.json` with the MHR joints/pose/shape parameters — all pushed to your browser's downloads at the end.

## Deliberate simplifications vs. the full official pipeline

- Detector: ViTDet-H only (the default). The alternative `sam3` detector is skipped — it requires cloning and installing an additional large model not needed to just get a mesh out on a T4.
- Mask-conditioned inference (SAM2 segmentor) is off by default — it requires a separately-hosted SAM2 checkpoint path and is an advanced/optional feature in the official demo, not needed for single-photo-to-mesh.
- The inline 3D preview render uses `pyrender`/OpenGL in headless mode, which is known to be flaky on fresh VMs. If it fails, it does **not** affect the actual mesh/JSON export below — that only needs plain NumPy arrays.


In [ ]:
# --- GPU check ---------------------------------------------------------------
!nvidia-smi

import torch

print(f"torch version:    {torch.__version__}")
print(f"torch CUDA build: {torch.version.cuda}")

assert torch.cuda.is_available(), (
    "No GPU visible to torch. Go to Runtime > Change runtime type > "
    "Hardware accelerator > GPU (T4), then Runtime > Run all."
)

props = torch.cuda.get_device_properties(0)
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1e9:.1f} GB total")


## Step 1 — clone the repo and install dependencies

Every cell in this section is safe to re-run: it checks what's already installed/cloned before doing anything, so re-running the whole notebook (e.g. after a runtime restart) will skip work that's already done.


In [ ]:
# --- clone facebookresearch/sam-3d-body ---------------------------------------
import os
import sys

REPO_DIR = "/content/sam-3d-body"

if not os.path.exists(REPO_DIR):
    print(f"Cloning facebookresearch/sam-3d-body into {REPO_DIR} ...")
    !git clone -q https://github.com/facebookresearch/sam-3d-body.git {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

assert os.path.isdir(os.path.join(REPO_DIR, "sam_3d_body")), "Clone looks incomplete - re-run this cell."
print("Repo ready, sys.path updated.")


In [ ]:
# --- core python dependencies (from the repo's INSTALL.md) --------------------
# Note: INSTALL.md lists a package called "chump" - that name actually resolves on PyPI to
# an unrelated Pushover API wrapper. The package this codebase almost certainly means is
# "chumpy" (the autodiff library used across the SMPL/MHR mesh-model family). We install both
# so neither possible intent is left unmet; both are tiny, so this costs nothing.
print("Installing core pip dependencies (long list, a couple of minutes)...")
!pip install -q pytorch-lightning pyrender opencv-python yacs scikit-image einops timm dill pandas rich hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils webdataset chump chumpy "networkx==3.2.1" roma joblib seaborn wandb appdirs appnope ffmpeg cython jsonlines pytest xtcocotools loguru optree fvcore black pycocotools tensorboard huggingface_hub

# Headless OpenGL/EGL system libs - needed only for the optional inline preview render later.
# Best-effort: this notebook does not depend on it succeeding.
!apt-get -qq update > /dev/null && apt-get -qq install -y libgl1-mesa-glx libgl1-mesa-dri libegl1-mesa libgles2-mesa > /dev/null 2>&1

import cv2, yacs, skimage, einops, timm, hydra, roma, fvcore, pycocotools  # noqa: F401 - sanity import
print("Core dependencies OK.")


In [ ]:
# --- detectron2 (required by the default ViTDet-H human detector) -------------
# Built from source pinned to commit a1ce2f9, exactly as documented in INSTALL.md. Building
# against whatever torch/CUDA Colab currently ships (rather than pinning a torch version here)
# is deliberate: Colab's preinstalled torch changes over time, and --no-build-isolation
# --no-deps compiles detectron2's CUDA extensions against whatever is already installed.
try:
    import detectron2
    print(f"detectron2 {detectron2.__version__} already installed, skipping build.")
except ImportError:
    print("Building detectron2 from source (pinned commit a1ce2f9)...")
    print("This compiles CUDA extensions and can take 5-10 minutes. If it fails with an nvcc")
    print("or CUDA-version-mismatch error, see the troubleshooting section of the companion doc.")
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps
    import detectron2
    print(f"detectron2 {detectron2.__version__} ready.")


In [ ]:
# --- MoGe2 (optional field-of-view estimator, on by default in the official pipeline) ------
# If this install fails for any reason we don't hard-fail the notebook: SAM 3D Body falls back
# to a default FOV heuristic. That mainly affects the estimated absolute camera scale/depth of
# the mesh, not the recovered body pose or shape (see the companion doc for the detail).
HAVE_MOGE = False
try:
    import moge  # noqa: F401
    HAVE_MOGE = True
    print("MoGe2 already installed, skipping.")
except ImportError:
    print("Installing MoGe2 (optional)...")
    !pip install -q git+https://github.com/microsoft/MoGe.git
    try:
        import moge  # noqa: F401
        HAVE_MOGE = True
        print("MoGe2 installed.")
    except ImportError as e:
        print(f"MoGe2 install did not produce an importable package ({e}).")
        print("Continuing without it - will use the default FOV heuristic instead.")

print(f"HAVE_MOGE = {HAVE_MOGE}")


## Step 2 — download model checkpoints

`MODEL_VARIANT` picks the backbone: `"dinov3"` (840M params, best accuracy, ~2.1 GB checkpoint, used by the official demo notebook by default) or `"vith"` (631M params, ~1.7 GB, a bit lighter if you hit memory/time pressure).

The cell below tries the official gated repo first (if you provide a token), and falls back to the ungated mirror if access hasn't been granted yet. `snapshot_download` caches to disk, so re-running this cell after the first successful run is a fast no-op.


In [ ]:
# --- Hugging Face auth + checkpoint download -----------------------------------
import getpass
from huggingface_hub import login, snapshot_download
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

MODEL_VARIANT = "dinov3"  # or "vith"
OFFICIAL_REPO = f"facebook/sam-3d-body-{MODEL_VARIANT}"
MIRROR_REPO = f"jetjodh/sam-3d-body-{MODEL_VARIANT}"

token = getpass.getpass(
    "Paste a Hugging Face access token (https://huggingface.co/settings/tokens) if you have "
    "one and have requested access to the official repo, or press Enter to skip straight to "
    "the ungated mirror: "
).strip()

if token:
    login(token=token)
    print("Logged in to Hugging Face.")
else:
    print("No token given - skipping login, will try the mirror repo directly.")

del token  # never keep the token value around longer than needed


def try_download(repo_id):
    print(f"Attempting download of {repo_id} ...")
    local_dir = snapshot_download(repo_id=repo_id)
    print(f"OK: {repo_id} cached at {local_dir}")
    return repo_id


HF_REPO_ID = None
try:
    HF_REPO_ID = try_download(OFFICIAL_REPO)
except (GatedRepoError, HfHubHTTPError) as e:
    print(f"Official repo {OFFICIAL_REPO} not accessible yet ({e.__class__.__name__}).")
    print(f"Falling back to mirror {MIRROR_REPO}.")

if HF_REPO_ID is None:
    try:
        HF_REPO_ID = try_download(MIRROR_REPO)
    except (GatedRepoError, HfHubHTTPError) as e:
        raise RuntimeError(
            f"Could not download weights from either {OFFICIAL_REPO} or {MIRROR_REPO}. "
            "Request access at the HF links in the first cell, wait for approval, and re-run "
            "this cell (or check whether the mirror repo id has changed)."
        ) from e

print(f"\nUsing checkpoint repo: {HF_REPO_ID}")
print("Note: the ViTDet-H detector checkpoint (~2.8 GB) and, if enabled, MoGe2 (~1.3 GB) "
      "download automatically the first time the estimator is built in Step 4 below.")


## Step 3 — provide a photo

Upload a clear, mostly-frontal photo with the full body visible (best results). If you don't upload anything, this cell falls back to the sample photo bundled with the official repo (`notebook/images/dancing.jpg`) so you can still verify the pipeline end-to-end.


In [ ]:
# --- image input ----------------------------------------------------------------
import os
from google.colab import files

UPLOAD_DIR = "/content/input"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Choose a photo to upload (a clear, mostly-frontal, full-body photo works best).")
print("Cancel/close the picker with nothing selected to use a bundled sample photo instead.")

try:
    uploaded = files.upload()
except Exception as e:
    print(f"Upload widget did not return a file ({e.__class__.__name__}); using sample photo.")
    uploaded = {}

if uploaded:
    fname = next(iter(uploaded))
    IMAGE_PATH = os.path.join(UPLOAD_DIR, fname)
    with open(IMAGE_PATH, "wb") as f:
        f.write(uploaded[fname])
    print(f"Using uploaded image: {IMAGE_PATH}")
else:
    IMAGE_PATH = os.path.join(UPLOAD_DIR, "dancing_sample.jpg")
    if not os.path.exists(IMAGE_PATH):
        !wget -q -O {IMAGE_PATH} https://raw.githubusercontent.com/facebookresearch/sam-3d-body/main/notebook/images/dancing.jpg
    print(f"No upload detected - using bundled sample image: {IMAGE_PATH}")

assert os.path.exists(IMAGE_PATH) and os.path.getsize(IMAGE_PATH) > 0, "Image file is missing or empty."


## Step 4 — build the estimator and run inference


In [ ]:
# --- build the SAM3DBodyEstimator ------------------------------------------------
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from notebook.utils import setup_sam_3d_body

fov_name = "moge2" if HAVE_MOGE else ""
if not HAVE_MOGE:
    print("Proceeding without MoGe2 - using the model's default FOV heuristic.")

estimator = setup_sam_3d_body(
    hf_repo_id=HF_REPO_ID,
    detector_name="vitdet",
    fov_name=fov_name,
)

print(f"\nEstimator ready. Mesh topology has {estimator.faces.shape[0]} faces.")


In [ ]:
# --- run inference on the chosen photo --------------------------------------------
outputs = estimator.process_one_image(IMAGE_PATH)

print(f"Detected {len(outputs)} people in {IMAGE_PATH}")
if outputs:
    print("Per-person output keys:", list(outputs[0].keys()))
    print("Vertices per person:", outputs[0]["pred_vertices"].shape)
else:
    print("No person detected. Try a clearer, more front-facing photo with the full body ")
    print("visible, better lighting, or less extreme cropping, then re-run this cell.")

assert len(outputs) > 0, "No person detected in the image - see message above."


## Step 5 — preview (optional, best-effort)

This renders the mesh overlaid on your photo plus a side view, purely for a sanity-check preview inline. It uses `pyrender`'s headless OpenGL/EGL backend, which is known to be flaky on fresh Linux VMs. **If this cell errors, skip it and move on** — the actual mesh/JSON export in the next section only needs the raw NumPy arrays already sitting in `outputs`, not a working GL context.


In [ ]:
# --- optional inline preview render -------------------------------------------------
try:
    import cv2
    import matplotlib.pyplot as plt
    from tools.vis_utils import visualize_sample_together

    img_bgr = cv2.imread(IMAGE_PATH)
    rend_img = visualize_sample_together(img_bgr, outputs, estimator.faces)
    rend_rgb = cv2.cvtColor(rend_img.astype("uint8"), cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(20, 6))
    plt.imshow(rend_rgb)
    plt.axis("off")
    plt.title("original | 2D keypoints | mesh overlay | side view")
    plt.show()
except Exception as e:
    print(f"Preview rendering failed ({e.__class__.__name__}: {e}).")
    print("This is a known headless-OpenGL/EGL issue and does NOT affect the mesh export below ")
    print("- it only affects this inline preview image. See the troubleshooting doc.")


## Step 6 — export mesh + joints/pose, download

For each detected person, writes:
- `..._mesh_NNN.obj` — the mesh (vertices + triangle faces), written by a small dependency-free OBJ writer defined below (no `pyrender`/GL dependency, so it works even if Step 5 failed).
- `..._mesh_NNN.glb` — best-effort binary glTF via `trimesh` (already installed as a `pyrender` dependency); skipped with a clear message if it fails, the `.obj` is unaffected.
- `..._pose_NNN.json` — MHR joints, pose parameters (body/hand/global rotation), shape/scale parameters, 2D/3D keypoints, focal length and camera translation.

The vertex transform (`pred_vertices + pred_cam_t`, then a 180-degree rotation about X) exactly reproduces what `sam_3d_body.visualization.renderer.Renderer.vertices_to_trimesh()` does internally (verified from source, `renderer.py` lines 260-286) so the exported mesh matches the orientation you'd see in the official demo's own PLY export.


In [ ]:
# --- export .obj / .glb / _pose.json and download -----------------------------------
import json
import os
import numpy as np
from google.colab import files

OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)
image_stub = os.path.splitext(os.path.basename(IMAGE_PATH))[0]


def sam3d_world_vertices(pred_vertices, pred_cam_t):
    """Reproduce Renderer.vertices_to_trimesh()'s transform: vertices + camera translation,
    then a 180-degree rotation about the X axis, i.e. (x, y, z) -> (x, -y, -z)."""
    v = (pred_vertices + pred_cam_t).copy()
    v[:, 1] *= -1
    v[:, 2] *= -1
    return v


def write_obj(path, vertices, faces):
    """Dependency-free Wavefront OBJ writer. faces are 0-indexed triangle vertex ids."""
    vertices = np.asarray(vertices, dtype=np.float64)
    faces = np.asarray(faces, dtype=np.int64)
    with open(path, "w", encoding="utf-8") as f:
        f.write("# SAM 3D Body mesh export (ARES pipeline)\n")
        f.write(f"# {len(vertices)} vertices, {len(faces)} faces\n")
        for v in vertices:
            f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
        for face in faces:
            f.write(f"f {face[0] + 1} {face[1] + 1} {face[2] + 1}\n")


def to_json_safe(obj):
    """Recursively convert numpy types to plain python for json.dump."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_json_safe(v) for v in obj]
    return obj


downloaded_files = []
for pid, person in enumerate(outputs):
    world_verts = sam3d_world_vertices(person["pred_vertices"], person["pred_cam_t"])

    obj_path = os.path.join(OUT_DIR, f"{image_stub}_mesh_{pid:03d}.obj")
    write_obj(obj_path, world_verts, estimator.faces)
    downloaded_files.append(obj_path)
    print(f"Wrote {obj_path}")

    try:
        import trimesh
        tmesh = trimesh.Trimesh(vertices=world_verts, faces=estimator.faces, process=False)
        glb_path = os.path.join(OUT_DIR, f"{image_stub}_mesh_{pid:03d}.glb")
        tmesh.export(glb_path)
        downloaded_files.append(glb_path)
        print(f"Wrote {glb_path}")
    except Exception as e:
        print(f"GLB export skipped ({e.__class__.__name__}: {e}); .obj above is unaffected.")

    pose_record = {k: v for k, v in person.items() if k != "mask"}
    pose_path = os.path.join(OUT_DIR, f"{image_stub}_pose_{pid:03d}.json")
    with open(pose_path, "w", encoding="utf-8") as f:
        json.dump(to_json_safe(pose_record), f, indent=2)
    downloaded_files.append(pose_path)
    print(f"Wrote {pose_path}")

print(f"\n{len(downloaded_files)} file(s) ready. Starting downloads ")
print("(check your browser's download prompts/permissions if nothing happens)...")
for path in downloaded_files:
    files.download(path)


## Next steps

- Full write-up, troubleshooting steps, and how this fits into the ARES pipeline: `ares/docs/sam3d-body-colab.md`.
- Official repo (for updates/newer checkpoints): https://github.com/facebookresearch/sam-3d-body
- Zero-setup hosted alternative: https://fal.ai/models/fal-ai/sam-3/3d-body (~$0.02/generation).
